## Inicializacion

Antes de comenzar, importamos las dependencias necesarias y definimos una funcion auxiliar para interactuar con la Messages API de Anthropic.

In [ ]:
import re
from anthropic import Anthropic

client = Anthropic()

def get_completion(prompt: str, system: str = "", prefill=None, messages=None) -> str:
    
    if messages is None:
        messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})

    kwargs = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        kwargs["system"] = system

    response = client.messages.create(**kwargs)
    return response.content[0].text


## Estructura de una llamada a la Messages API

Una llamada a la Messages API requiere como minimo los siguientes argumentos:

- **model**: el modelo a utilizar (por ejemplo, `claude-haiku-4-5-20251001`)
- **max_tokens**: el numero maximo de tokens que puede contener la respuesta
- **messages**: la lista de mensajes que conforman la conversacion

Adicionalmente, se pueden incluir parametros opcionales como:

- **system**: el system prompt que define el comportamiento general del modelo
- **temperature**: controla la aleatoriedad de las respuestas (0.0 = determinista, 1.0 = mas creativo)

**Reglas importantes sobre `messages`:**

1. Los roles `user` y `assistant` deben alternarse estrictamente. No se pueden tener dos mensajes consecutivos del mismo rol.
2. La conversacion siempre debe comenzar con un mensaje de rol `user`.

Violar cualquiera de estas reglas producira un error en la API.

In [ ]:
SYSTEM = "Eres un asistente experto en historia. Tus respuestas deben ser claras, concisas y bien fundamentadas."
PROMPT = "Explica brevemente las causas principales de la Revolucion Francesa."

print(get_completion(PROMPT, SYSTEM))

### Que es el System Prompt?

El system prompt le indica al modelo como debe comportarse, que tipo de respuestas debe generar y bajo que contexto debe operar. En el ejemplo anterior, le indicamos que actuara como un experto en historia con respuestas claras y fundamentadas.

El system prompt es una herramienta poderosa porque permite moldear el comportamiento del modelo sin necesidad de repetir instrucciones en cada mensaje del usuario. Esto mejora tanto la consistencia como la eficiencia de las respuestas.

## La importancia de dar instrucciones claras

Claude es como una persona que recien se incorpora a un nuevo trabajo: si se le dan instrucciones claras y detalladas, realizara la tarea de manera casi perfecta. Pero si las instrucciones son vagas o ambiguas, el resultado puede no ser el esperado.

Un prompt bien construido marca una gran diferencia en la calidad de la respuesta. Por ejemplo, podemos hacer una pregunta abierta sin system prompt y observar como responde el modelo.

In [ ]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos?"
SYSTEM = ""

print(get_completion(PROMPT, SYSTEM))

Se puede observar que el modelo ofrece varias opciones o matiza su respuesta. Si necesitamos una respuesta unica y definitiva, debemos ser mas especificos en nuestra instruccion. La claridad y precision en el prompt es clave para obtener el tipo de respuesta que buscamos.

In [ ]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos? Elige solo uno y justifica tu eleccion brevemente."
SYSTEM = ""

print(get_completion(PROMPT, SYSTEM))

## Prompt Templates

Una tecnica muy util es el uso de **prompt templates**: plantillas de prompts donde solo se cambian pequenas partes (variables) para reutilizar la misma estructura con diferentes datos de entrada. Esto es especialmente valioso cuando se necesita ejecutar la misma tarea sobre multiples conjuntos de datos.

En Python, se pueden usar f-strings o el metodo `.format()` para insertar variables dinamicamente en el prompt.

In [ ]:
DATASET_1 = [
    {"nombre": "Alice", "edad": 30, "ciudad": "Madrid"},
    {"nombre": "Bob", "edad": 25, "ciudad": "Barcelona"},
    {"nombre": "Charlie", "edad": 35, "ciudad": "Buenos Aires"}
]

DATASET_2 = [
    {"producto": "Laptop", "precio": 999.99, "stock": 10},
    {"producto": "Smartphone", "precio": 499.99, "stock": 20},
    {"producto": "Tablet", "precio": 299.99, "stock": 15}
]

SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."
PROMPT_TEMPLATE = "Por favor analiza el siguiente conjunto de datos y proporciona un resumen detallado: {datos}"

print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT_TEMPLATE.format(datos=str(DATASET_1)), SYSTEM))

print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT_TEMPLATE.format(datos=str(DATASET_2)), SYSTEM))

## Etiquetas XML para estructurar prompts

Claude fue entrenado con datos que incluyen etiquetas XML, por lo que las comprende y utiliza de manera natural. Encerrar datos dentro de etiquetas XML como `<datos></datos>` ofrece varias ventajas:

1. **Separacion clara** entre instrucciones y datos, lo que reduce la ambiguedad
2. **Mejor comprension** del prompt por parte del modelo, especialmente cuando los datos son complejos o contienen ruido
3. **Respuestas estructuradas**: se le puede pedir al modelo que responda dentro de etiquetas XML especificas, lo que facilita la extraccion programatica de la informacion relevante

In [ ]:
SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."

PROMPT = f"Por favor analiza el siguiente conjunto de datos y proporciona un resumen detallado: <datos>{str(DATASET_1)}</datos>"
print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT, SYSTEM))

PROMPT = f"Por favor analiza el siguiente conjunto de datos y proporciona un resumen detallado: <datos>{str(DATASET_2)}</datos>"
print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT, SYSTEM))

### Prefilling: poner palabras en la boca de Claude

Ademas de usar etiquetas XML en la entrada, se puede forzar al modelo a responder dentro de etiquetas XML especificas mediante la tecnica de **prefilling**. Esta tecnica consiste en iniciar la respuesta del asistente con un texto predefinido (por ejemplo, la etiqueta de apertura `<resumen>`), de modo que el modelo continue generando desde ese punto.

Esto es util para:

- Garantizar un formato de salida consistente y predecible
- Facilitar el parseo automatico de las respuestas
- Guiar al modelo hacia el tipo de contenido deseado

In [ ]:
SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."
PREFILL = "<resumen>"

PROMPT = f"Analiza el siguiente conjunto de datos y proporciona un resumen detallado dentro de etiquetas <resumen>: <datos>{str(DATASET_1)}</datos>"
print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

PROMPT = f"Analiza el siguiente conjunto de datos y proporciona un resumen detallado dentro de etiquetas <resumen>: <datos>{str(DATASET_2)}</datos>"
print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

## Chain of Thought (Pensamiento paso a paso)

Incluso con un buen system prompt y contexto adecuado, el modelo puede cometer errores en tareas complejas que requieren razonamiento en multiples pasos. Una tecnica efectiva para mejorar la calidad de las respuestas es el **chain of thought**: pedirle al modelo que razone paso a paso antes de dar su respuesta final.

Es equivalente a darle a una persona no solo la tarea, sino tambien un procedimiento detallado de como abordarla. Al descomponer el razonamiento en pasos explicitos, el modelo tiende a producir respuestas mas precisas y fundamentadas.

In [ ]:
SYSTEM = """Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras).
Antes de responder, sigue estos pasos:
1. Analiza el numero de entradas y sus atributos
2. Identifica patrones o correlaciones entre los atributos
3. Genera un resumen detallado de los hallazgos
Responde dentro de etiquetas <resumen>."""

PREFILL = "<resumen>"

PROMPT = f"Analiza el siguiente conjunto de datos paso a paso: <datos>{str(DATASET_1)}</datos>"
print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

PROMPT = f"Analiza el siguiente conjunto de datos paso a paso: <datos>{str(DATASET_2)}</datos>"
print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

## Few-Shot Prompting

El **few-shot prompting** consiste en proporcionarle al modelo uno o mas ejemplos del comportamiento deseado antes de hacerle la solicitud real. Estos ejemplos pueden incluirse de dos formas: directamente en el texto del prompt, o como mensajes previos en la secuencia de conversacion (alternando roles `user` y `assistant`).

La nomenclatura es la siguiente:

- **Zero-shot**: sin ejemplos previos, solo la instruccion
- **One-shot**: un unico ejemplo antes de la solicitud
- **Few-shot**: varios ejemplos
- **N-shot**: n ejemplos en general

Cuantos mas ejemplos relevantes y representativos se proporcionen, mejor entendera el modelo el patron esperado.

In [ ]:
PROMPT = """Por favor completa la conversacion escribiendo la siguiente linea. Tu eres el Asistente:

<conversacion>
Usuario: Buenos dias, como se encuentra?
Asistente: Muy bien, gracias por preguntar. En que puedo ayudarle hoy?
Usuario: Me gustaria saber sobre cursos de programacion disponibles.
</conversacion>
"""

print(get_completion(PROMPT))

# Nota: lo anterior es equivalente a usar la secuencia de mensajes directamente:
# messages = [
#     {"role": "user", "content": "Buenos dias, como se encuentra?"},
#     {"role": "assistant", "content": "Muy bien, gracias por preguntar. En que puedo ayudarle hoy?"},
#     {"role": "user", "content": "Me gustaria saber sobre cursos de programacion disponibles."}
# ]

## Reducir alucinaciones

En ocasiones, el modelo puede generar informacion incorrecta o inventada. Esto no se debe a un fallo, sino a su tendencia a intentar ser util incluso cuando no tiene informacion suficiente para responder con precision. A este fenomeno se le conoce como **alucinacion** (hallucination).

Para mitigar este problema, existen varias estrategias:

- **Dar una salida explicita**: indicarle al modelo que si no sabe algo, responda honestamente que no lo sabe en lugar de inventar una respuesta
- **Reducir la temperature**: valores mas bajos (cercanos a 0) producen respuestas mas conservadoras y menos propensas a la invencion
- **Pedir evidencia o justificacion**: solicitar que cite fuentes o fundamente sus afirmaciones

In [ ]:
SYSTEM = "Eres un asistente de conocimiento general. Responde con precision y honestidad."
PROMPT = "Cual es la poblacion exacta de la ciudad de Atlantida? Solo responde si sabes la respuesta con certeza. De lo contrario, indica que no tienes esa informacion."

print(get_completion(PROMPT, SYSTEM))

## Guia de estructura para prompts complejos

Cuando se trabaja con tareas complejas, es recomendable seguir una estructura organizada para construir el prompt. A continuacion se presenta una guia con los elementos clave, en orden sugerido.

---

### Elemento 1: Rol `user`

La llamada a la Messages API siempre debe comenzar con un mensaje de rol `user` en el array de mensajes.

### Elemento 2: Contexto de la tarea

Proporcionar al modelo el contexto sobre el rol que debe asumir, los objetivos generales y la tarea a realizar. Es recomendable colocar el contexto al inicio del prompt.

**Ejemplo:**
```
Actuaras como un asistente de atencion al cliente creado por la empresa X. Tu objetivo es resolver dudas de los usuarios que visitan el sitio web de la empresa.
```

### Elemento 3: Contexto de tono

Si es relevante para la interaccion, indicar al modelo que tono debe utilizar. Este elemento no siempre es necesario, depende de la tarea.

**Ejemplo:**
```
Debes mantener un tono profesional y amigable en todo momento.
```

### Elemento 4: Descripcion detallada de la tarea y reglas

Expandir las tareas especificas que el modelo debe realizar, asi como las reglas que debe seguir. Aqui tambien se le puede dar una "salida" para cuando no tenga la respuesta.

**Ejemplo:**
```
Reglas importantes para la interaccion:
- Mantente siempre en el rol asignado
- Si no estas seguro de como responder, di: "Disculpa, no entendi tu pregunta. Podrias reformularla?"
- Si alguien pregunta algo fuera de tema, redirige amablemente hacia el proposito de la conversacion
```

### Elemento 5: Ejemplos (few-shot)

Proporcionar al modelo al menos un ejemplo de la respuesta ideal que se espera. Encerrar los ejemplos en etiquetas XML. Si se proporcionan multiples ejemplos, cada uno debe estar en su propio par de etiquetas.

Los ejemplos son probablemente la herramienta mas efectiva para lograr que el modelo se comporte como se desea. Se recomienda incluir ejemplos de casos comunes y casos limite.

**Ejemplo:**
```
Aqui hay un ejemplo de como responder en una interaccion estandar:

<ejemplo>
Usuario: Hola, que haces y como fuiste creado?
Asistente: Hola! Soy un asistente virtual creado por la empresa X para ayudarte con tus consultas. En que puedo asistirte hoy?
</ejemplo>
```

### Elemento 6: Datos de entrada

Si el modelo necesita procesar datos especificos, incluirlos dentro de etiquetas XML relevantes. Se pueden incluir multiples bloques de datos, cada uno en sus propias etiquetas.

**Ejemplo:**
```
Aqui esta el historial de la conversacion previa entre el usuario y tu:
<historial>
{historial_conversacion}
</historial>

Aqui esta la pregunta actual del usuario:
<pregunta>
{pregunta_usuario}
</pregunta>
```

### Elemento 7: Solicitud inmediata

"Recordarle" al modelo exactamente que debe hacer para cumplir con la tarea. Es mejor colocar esto hacia el final del prompt, ya que produce mejores resultados que ponerlo al principio. Tambien es buena practica colocar la consulta del usuario cerca del final.

**Ejemplo:**
```
Como respondes a la pregunta del usuario?
```

### Elemento 8: Precognicion (pensamiento paso a paso)

Para tareas con multiples pasos, indicar al modelo que piense paso a paso antes de dar una respuesta. En ocasiones es necesario usar frases como "Antes de dar tu respuesta..." para asegurar que el modelo realice el razonamiento previo.

No es necesario en todos los prompts, pero cuando se incluye, es mejor colocarlo hacia el final y justo despues de la solicitud inmediata.

**Ejemplo:**
```
Piensa en tu respuesta paso a paso antes de responder.
```

### Elemento 9: Formato de salida

Si se requiere un formato especifico en la respuesta, indicarlo claramente. Es mejor colocar esto hacia el final del prompt.

**Ejemplo:**
```
Coloca tu respuesta dentro de etiquetas <respuesta>.
```

### Elemento 10: Prefilling de la respuesta

Un espacio para iniciar la respuesta del modelo con texto predefinido que guie su comportamiento. Si se desea usar prefilling, debe colocarse en el rol `assistant` de la llamada a la API.

**Ejemplo:**
```
[Asistente]
```

---

### Resumen

No todos los elementos son necesarios para cada prompt. La estructura y los elementos a incluir dependen de la complejidad de la tarea. Sin embargo, para tareas complejas, seguir esta guia ayuda a obtener respuestas mas consistentes y de mayor calidad.

## Prompt Chaining: encadenar llamadas

Existen varios patrones que permiten mejorar sustancialmente la calidad de las respuestas del modelo. La idea fundamental es utilizar la salida de una llamada como entrada de la siguiente, creando una cadena de procesamiento. Los tres patrones mas comunes son:

1. **Revision**: generar una respuesta y luego pedirle al modelo que la revise y corrija
2. **Mejora iterativa**: generar un borrador y luego pedir que lo refine con instrucciones especificas
3. **Pipeline de procesamiento**: usar el resultado de un prompt como dato de entrada para otro prompt con una tarea completamente diferente

### Patron 1: Revision

El patron de revision consiste en pedirle al modelo que genere una respuesta y luego, en un segundo turno, que la revise y corrija. Esto es especialmente util cuando la tarea inicial es propensa a errores o alucinaciones, ya que el modelo puede detectar y corregir sus propios fallos al examinar su salida con ojos criticos.

In [ ]:
# Paso 1: El modelo genera una respuesta inicial
first_response = get_completion(
    "Genera una lista de 10 palabras poco comunes en espanol y su significado."
)
print("--- Respuesta inicial ---")
print(first_response)

# Paso 2: Pedirle que revise su propia respuesta
messages = [
    {"role": "user", "content": "Genera una lista de 10 palabras poco comunes en espanol y su significado."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": "Revisa la lista anterior. Verifica que todas las palabras existan realmente y que los significados sean correctos. Reemplaza cualquier palabra incorrecta o inventada."}
]

revision = get_completion("", messages=messages)
print("\n--- Respuesta revisada ---")
print(revision)

El modelo revisa su propia respuesta y corrige los errores que detecta. Sin embargo, existe un problema comun: si la respuesta original ya era correcta, el modelo a veces la modifica innecesariamente porque interpreta que se le esta pidiendo que cambie algo.

La solucion es **darle una salida explicita**: indicarle que si todo esta correcto, lo deje tal cual. De esta forma, el modelo solo modifica lo que realmente necesita correccion.

In [ ]:
# Instruccion de revision mejorada con salida explicita
revision_prompt = (
    "Revisa las palabras anteriores. Reemplaza las que no sean reales. "
    "Si TODAS son correctas, mantenlas tal cual -- no cambies palabras correctas."
)

messages = [
    {"role": "user", "content": "Genera una lista de 10 palabras poco comunes en espanol y su significado."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": revision_prompt}
]

revision_mejorada = get_completion("", messages=messages)
print(revision_mejorada)

### Patron 2: Mejora iterativa

Este patron consiste en generar un borrador inicial y luego pedirle al modelo que lo mejore con instrucciones especificas. La diferencia con la revision es que aqui no se busca corregir errores, sino elevar la calidad general del contenido. Es equivalente a un proceso de edicion real, donde un primer borrador se refina sucesivamente.

In [ ]:
# Paso 1: Generar un borrador inicial
borrador = get_completion("Escribe un parrafo corto explicando que es la inteligencia artificial.")
print("--- Borrador v1 ---")
print(borrador)

# Paso 2: Pedir una mejora especifica
messages = [
    {"role": "user", "content": "Escribe un parrafo corto explicando que es la inteligencia artificial."},
    {"role": "assistant", "content": borrador},
    {"role": "user", "content": "Mejora este texto. Agrega ejemplos concretos, usa un lenguaje mas preciso y haz que la conclusion sea mas impactante."}
]

version_mejorada = get_completion("", messages=messages)
print("\n--- Borrador v2 (mejorado) ---")
print(version_mejorada)

La segunda version suele ser notablemente mejor porque el modelo puede analizar lo que genero previamente y refinarlo con instrucciones especificas. La clave esta en dar directrices concretas sobre que aspectos mejorar (ejemplos, precision, estructura, etc.) en lugar de pedir simplemente "hazlo mejor".

### Patron 3: Pipeline de procesamiento

En este patron, el resultado de un prompt se utiliza como dato de entrada para otro prompt con una tarea completamente diferente. A diferencia de la revision o la mejora iterativa, aqui cada paso realiza una tarea distinta, formando un pipeline donde la informacion fluye de una etapa a la siguiente.

Esto es util para descomponer tareas complejas en pasos simples y especializados, donde cada paso se puede optimizar de forma independiente.

In [ ]:
texto_largo = """La doctora Maria Lopez presento su investigacion sobre cambio climatico 
en la conferencia de Berlin. El profesor Carlos Ruiz, de la Universidad de Buenos Aires, 
colaboro en el estudio junto con la ingeniera Ana Torres del MIT."""

# Paso 1: Extraer nombres propios del texto
PROMPT_EXTRACCION = f"""Extrae todos los nombres propios de personas del siguiente texto.
Devuelvelos como una lista separada por comas.
<texto>{texto_largo}</texto>"""

nombres = get_completion(PROMPT_EXTRACCION)
print("--- Nombres extraidos ---")
print(nombres)

# Paso 2: Usar los nombres extraidos como entrada para otra tarea
PROMPT_BIOGRAFIAS = f"""Escribe una biografia ficticia de un parrafo para cada una de las 
siguientes personas: {nombres}"""

biografias = get_completion(PROMPT_BIOGRAFIAS)
print("\n--- Biografias generadas ---")
print(biografias)

## Tool Use (Function Calling)

El uso de herramientas (tool use) permite que el modelo interactue con funciones externas. Es importante entender que **el modelo no ejecuta codigo directamente**. Lo que hace es generar una solicitud estructurada indicando que funcion quiere llamar y con que parametros. El desarrollador ejecuta esa funcion en su propio entorno y le devuelve el resultado al modelo para que continue la conversacion.

En esencia, tool use es una combinacion de dos tecnicas ya vistas:

- **Sustitucion**: insertar resultados de funciones dentro de prompts
- **Prompt chaining**: encadenar el resultado de vuelta al modelo

El proceso se divide en varios pasos que se describen a continuacion.

### Paso 1: Definir las herramientas en el System Prompt

El system prompt para tool use tiene dos partes: una seccion general que explica el mecanismo de llamada, y una seccion especifica que define las herramientas disponibles.

**Parte 1 -- Explicacion general (reutilizable para cualquier proyecto):**

Esta seccion le explica al modelo el mecanismo de llamada a herramientas: como debe formatear sus solicitudes cuando necesite usar una funcion.

In [ ]:
system_tools_general = """Tienes acceso a un conjunto de herramientas que puedes usar 
para responder la pregunta del usuario.

Cuando necesites usar una herramienta, escribe la llamada dentro de 
etiquetas <function_calls> con el siguiente formato:

<function_calls>
    <invoke name="nombre_herramienta">
        <parameter name="param1">valor1</parameter>
        <parameter name="param2">valor2</parameter>
    </invoke>
</function_calls>
"""

**Parte 2 -- Herramientas especificas (cambia segun el caso de uso):**

Aqui se definen las herramientas concretas disponibles, con su nombre, descripcion y parametros. La descripcion de cada herramienta es fundamental, ya que es lo que el modelo lee para decidir cuando usarla y cuando no.

In [ ]:
system_tools_specific = """
<tool_description>
    <tool_name>calculadora</tool_name>
    <description>
        Realiza operaciones aritmeticas basicas.
        Usar cuando se necesite calcular algo.
    </description>
    <parameters>
        <parameter>
            <name>primer_operando</name>
            <type>int</type>
            <description>Primer numero de la operacion</description>
        </parameter>
        <parameter>
            <name>segundo_operando</name>
            <type>int</type>
            <description>Segundo numero de la operacion</description>
        </parameter>
        <parameter>
            <name>operador</name>
            <type>str</type>
            <description>Operador aritmetico: +, -, *, /</description>
        </parameter>
    </parameters>
</tool_description>
"""

# Combinar ambas partes para formar el system prompt completo
system_prompt = system_tools_general + system_tools_specific

### Paso 2: Enviar un mensaje que requiera el uso de la herramienta

Se envia un mensaje del usuario cuya respuesta requiere el uso de la herramienta definida. El parametro `stop_sequences` es clave: hace que el modelo se detenga justo despues de generar la etiqueta de cierre `</function_calls>`, evitando que continue generando texto despues de la llamada.

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=system_prompt,
    messages=[{
        "role": "user",
        "content": "Cuanto es 25 * 17?"
    }],
    max_tokens=500,
    stop_sequences=["</function_calls>"]
)

### Paso 3: El modelo genera la llamada a la herramienta

El modelo no responde directamente con el resultado. En lugar de eso, genera una solicitud estructurada indicando que herramienta quiere usar y con que parametros. Primero razona sobre lo que necesita hacer y luego genera la llamada en el formato XML definido.

La salida del modelo se vera algo asi:

```
Necesito calcular 25 * 17. Usare la calculadora.

<function_calls>
    <invoke name="calculadora">
        <parameter name="primer_operando">25</parameter>
        <parameter name="segundo_operando">17</parameter>
        <parameter name="operador">*</parameter>
    </invoke>
```

El codigo del desarrollador detecta la presencia de `</function_calls>` en el `stop_reason` y sabe que hay una llamada a herramienta pendiente de ejecutar.

### Paso 4: Ejecutar la funcion localmente

El desarrollador extrae los parametros de la respuesta del modelo y ejecuta la funcion real en su propio entorno. El modelo nunca ejecuta codigo; solo indica que funcion llamar y con que argumentos.

In [ ]:
def calculadora(primer_operando, segundo_operando, operador):
    """Funcion real que ejecuta la operacion aritmetica."""
    operaciones = {
        "+": lambda a, b: a + b,
        "-": lambda a, b: a - b,
        "*": lambda a, b: a * b,
        "/": lambda a, b: a / b,
    }
    if operador not in operaciones:
        raise ValueError(f"Operador no soportado: {operador}")
    return operaciones[operador](primer_operando, segundo_operando)

# Extraer parametros de la respuesta del modelo y ejecutar
resultado = calculadora(25, 17, "*")
print(f"Resultado de la ejecucion local: {resultado}")

### Paso 5: Formatear el resultado

El resultado de la funcion debe empaquetarse en un formato XML especifico que el modelo reconoce. Notar que el texto comienza con `</function_calls>`: esto se debe a que el modelo se detuvo antes de generar esa etiqueta de cierre (por el `stop_sequences`), por lo que el desarrollador la agrega manualmente junto con el resultado.

In [ ]:
formatted_result = f"""
</function_calls>
<function_results>
<result>
<tool_name>calculadora</tool_name>
<stdout>
{resultado}
</stdout>
</result>
</function_results>
"""

print("Resultado formateado para devolver al modelo:")
print(formatted_result)

### Paso 6: Enviar el resultado de vuelta al modelo

Finalmente, se concatena la respuesta original del modelo con el resultado formateado y se envia todo de vuelta como continuacion de la conversacion. El modelo recibe el resultado de la herramienta y genera su respuesta final en lenguaje natural.

In [ ]:
# Concatenar la respuesta original del modelo + el resultado de la herramienta
full_response = response.content[0].text + formatted_result

messages = [
    {"role": "user", "content": "Cuanto es 25 * 17?"},
    {"role": "assistant", "content": full_response},
    {"role": "user", "content": "Presenta el resultado."}
]

final = client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=system_prompt,
    messages=messages,
    max_tokens=500
)

print("Respuesta final del modelo:")
print(final.content[0].text)